In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

results_dir = PROJECT_ROOT / "data" / "results"

sa = pd.read_csv(
    results_dir / "focal_self_assertion_scores.csv"
)

values = pd.read_csv(
    results_dir / "focal_self_assertion_values.csv"
)

print("Speeches:", len(sa))
print("Self-Assertion sentences:", len(values))

Speeches: 180
Self-Assertion sentences: 1230


In [8]:

# Block 1 — Build country-level analysis dataset

# Define the 9 predefined regional case pairs
region_map = {
    "USA": "Modern West", "GBR": "Modern West",
    "CHN": "East Asia", "JPN": "East Asia",
    "SAU": "Arab Region", "JOR": "Arab Region",
    "CHL": "Latin America", "COL": "Latin America",
    "IND": "South Asia", "PAK": "South Asia",
    "GHA": "Sub-Saharan Africa", "KEN": "Sub-Saharan Africa",
    "IDN": "Southeast Asia", "THA": "Southeast Asia",
    "KAZ": "Central Asia", "TJK": "Central Asia",
    "POL": "Eastern Europe", "ROU": "Eastern Europe"
}

# Identify the 20 human-value columns
metadata_cols = {
    "iso3", "year", "filename", "sentence_id",
    "sentence_text", "matched_terms", "n_matches"
}
value_cols = [c for c in values.columns if c not in metadata_cols]

assert len(value_cols) == 20, "Expected 20 value columns."
assert sa.groupby("iso3")["year"].nunique().eq(10).all()
assert set(sa["iso3"]) == set(region_map)

# Country-level Self-Assertion prevalence
country_sa = (
    sa.groupby("iso3")
    .agg(
        sa_per_1000=("sa_per_1000", "mean"),
        n_speeches=("year", "size"),
        n_sa_sentences=("n_matches", "sum")
    )
)

# Average values within each country-year first
yearly_values = (
    values.groupby(["iso3", "year"])[value_cols]
    .mean()
    .reset_index()
)

# Then average available years equally within each country
country_values = (
    yearly_values.groupby("iso3")[value_cols]
    .mean()
)

# Count years with Self-Assertion value observations
n_value_years = (
    yearly_values.groupby("iso3")["year"]
    .nunique()
    .rename("n_value_years")
)
country_sa["n_sa_sentences"] = values.groupby("iso3").size()
# Combine into one dataset
country_df = (
    country_sa
    .join(country_values)
    .join(n_value_years)
    .reset_index()
)

country_df["region"] = country_df["iso3"].map(region_map)

# Quick verification
print("Countries:", len(country_df))
print("Values:", len(value_cols))
print("Total speeches:", country_df["n_speeches"].sum())
print("Total SA sentences:", country_df["n_sa_sentences"].sum())

display(
    country_df[
        ["iso3", "region", "sa_per_1000",
         "n_sa_sentences", "n_value_years"]
    ].sort_values(["region", "iso3"]).round(3)
)


Countries: 18
Values: 20
Total speeches: 180
Total SA sentences: 1230


,iso3,region,sa_per_1000,n_sa_sentences,n_value_years
7,JOR,Arab Region,1.844,30,10
14,SAU,Arab Region,4.205,82,10
9,KAZ,Central Asia,3.210,75,10
16,TJK,Central Asia,2.688,63,10
1,CHN,East Asia,4.440,117,10
8,JPN,East Asia,1.987,53,10
12,POL,Eastern Europe,2.698,81,10
13,ROU,Eastern Europe,2.072,34,10
0,CHL,Latin America,2.660,79,10
2,COL,Latin America,3.539,93,10


In [9]:

# Block 2 — Aggregate country scores into regional case-pair means

# Define a consistent display order
region_order = [
    "Modern West",
    "East Asia",
    "Arab Region",
    "Latin America",
    "South Asia",
    "Sub-Saharan Africa",
    "Southeast Asia",
    "Central Asia",
    "Eastern Europe"
]

# Average the two countries equally within each region
region_df = (
    country_df
    .groupby("region")[["sa_per_1000"] + value_cols]
    .mean()
    .reindex(region_order)
    .reset_index()
)

# Confirm that each region contains exactly two countries
region_counts = country_df.groupby("region")["iso3"].nunique()

assert len(region_df) == 9
assert region_counts.eq(2).all()

print("Regional case pairs:", len(region_df))

# Display the regional averages for the RQ1 variables
display(
    region_df[
        [
            "region",
            "sa_per_1000",
            "Self-direction: action",
            "Self-direction: thought"
        ]
    ].round(3)
)


Regional case pairs: 9


,region,sa_per_1000,Self-direction: action,Self-direction: thought
0,Modern West,2.736,0.197,0.155
1,East Asia,3.213,0.128,0.118
2,Arab Region,3.025,0.104,0.059
3,Latin America,3.100,0.171,0.121
4,South Asia,1.808,0.123,0.068
5,Sub-Saharan Africa,2.132,0.124,0.072
6,Southeast Asia,2.053,0.117,0.063
7,Central Asia,2.949,0.047,0.051
8,Eastern Europe,2.385,0.113,0.078


In [10]:

# Block 3 — RQ1: Spearman correlations at two aggregation levels

autonomy_values = [
    "Self-direction: action",
    "Self-direction: thought"
]

results = []

for level, df in [
    ("Country", country_df),
    ("Regional pair", region_df)
]:
    for value in autonomy_values:
        rho, p = spearmanr(
            df["sa_per_1000"],
            df[value],
            nan_policy="omit"
        )

        results.append({
            "Level": level,
            "Value": value,
            "N": df[["sa_per_1000", value]].dropna().shape[0],
            "Spearman rho": rho,
            "p-value (descriptive)": p
        })

rq1_results = pd.DataFrame(results)

display(rq1_results.round(3))


,Level,Value,N,Spearman rho,p-value (descriptive)
0,Country,Self-direction: action,18,-0.040,0.874
1,Country,Self-direction: thought,18,-0.098,0.699
2,Regional pair,Self-direction: action,9,0.167,0.668
3,Regional pair,Self-direction: thought,9,0.267,0.488
